## Making my own GPT

In [1]:
import torch
device = torch.device("cuda"if torch.cuda.is_available()else "cpu")

from datasets import load_dataset
data = load_dataset("roneneldan/TinyStories",)

c:\Users\suees\skill\cohort 4.0\AI\Learning-AI\gptenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Tokenize the prompt
from transformers import AutoModel,AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token # gt2 doesnt have padiing token makin that as eos token.
gpt_2 = AutoModel.from_pretrained("gpt2")

tokens = tokenizer(data["train"]['text'][:6],return_tensors="pt",padding=True,truncation=True) #tokens form user prompt as per gpt-2

Vector_embeddings = gpt_2.get_input_embeddings()

embeddings = Vector_embeddings(tokens["input_ids"]) # embeddings from the gpt-2 model


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 6596.99it/s]


In [3]:
# Tokens and the vector representation of it
print(f"Glimpse of tokens of the data :- {tokens["input_ids"][0][0:10]}")
print(f"length tokens of the data :- {len(tokens["input_ids"][0])}")

print("  ")
print("-------------------------------------")
print("  ")

print(f" shape of the Embeddings of the data :- {embeddings.shape}")
print(f" glimpse of the embeddigs :- {embeddings[0][0][0:5]}")

Glimpse of tokens of the data :- tensor([ 3198,  1110,    11,   257,  1310,  2576,  3706, 20037,  1043,   257])
length tokens of the data :- 212
  
-------------------------------------
  
 shape of the Embeddings of the data :- torch.Size([6, 212, 768])
 glimpse of the embeddigs :- tensor([-0.0672,  0.0269,  0.0795, -0.0823, -0.1724], grad_fn=<SliceBackward0>)


In [4]:
# add positional encoding to the Vector Embeddings
import torch
import numpy as np
def positional_encoding(length_of_ids,embeddings_lengths):
    # this takes the same shape as vector embeddings gives u positional embeddings for it

    pos = length_of_ids
    d = embeddings_lengths
    pe = torch.zeros((1,pos,d))
    for i in range(pos):
        for j in range(d//2):
            pe[0,i,2*j] = np.sin(i / 10000 ** (2 * j / d ))
            pe[0,i,2*j+1] = np.cos(i / 10000 ** (2 * j/ d ))
    return pe


In [5]:
#positional encodings
pe = positional_encoding(embeddings.shape[1],embeddings.shape[2])


In [6]:
embeddings.shape

torch.Size([6, 212, 768])

In [7]:
pe.shape

torch.Size([1, 212, 768])

In [8]:
# making pe ([200,298,768])

pe = pe.repeat(embeddings.shape[0],1,1)
pe.shape

torch.Size([6, 212, 768])

In [9]:
# final input embeddings = vector_embedding + positional encoding
input_embeddings = embeddings + pe

In [10]:

print(f" shape of the input embeddings of the user prompt :- {input_embeddings.shape}")
print(f" glimpse of the input embeddigs :- {embeddings[0][0][0:5]}")

 shape of the input embeddings of the user prompt :- torch.Size([6, 212, 768])
 glimpse of the input embeddigs :- tensor([-0.0672,  0.0269,  0.0795, -0.0823, -0.1724], grad_fn=<SliceBackward0>)


## Pre Attention Done

In [11]:

r1 = input_embeddings.clone()



In [12]:



#init the wieghts

w_k = torch.rand(input_embeddings.shape[-1],input_embeddings.shape[-1]).requires_grad_(True)
w_v = torch.rand(input_embeddings.shape[-1],input_embeddings.shape[-1]).requires_grad_(True)
w_q = torch.rand(input_embeddings.shape[-1],input_embeddings.shape[-1]).requires_grad_(True)

# calculating the Query , Key ,Value

Q = input_embeddings@w_q
K = input_embeddings@w_k
V = input_embeddings@w_v

num_head = 12
if 768%num_head ==0:
    dim = int(768/num_head)
else:
    raise ValueError("chose diff num_head")

Q = Q.reshape(input_embeddings.shape[0],input_embeddings.shape[1],num_head,dim).transpose(-2,1)
K = K.reshape(input_embeddings.shape[0],input_embeddings.shape[1],num_head,dim).transpose(-2,1)
V = V.reshape(input_embeddings.shape[0],input_embeddings.shape[1],num_head,dim).transpose(-2,1)

# Attention Score
scores = (Q@K.transpose(-2,-1))/np.sqrt(dim)

mask = torch.triu(torch.ones(scores.shape[-1],scores.shape[-1]), diagonal=1).bool()
scores = scores.masked_fill(mask, float("-inf"))

Attention_Score = torch.nn.functional.softmax(scores,dim=-1)@V
b,n,t,h = V.shape
Attention_out = Attention_Score.permute(0, 2, 1, 3).contiguous().reshape(b, t, n*h)
Attention_out += r1

Attention_out_rms = torch.nn.functional.rms_norm(Attention_out,(Attention_out.shape[-1],))
Attention_rms = Attention_out_rms



In [13]:
print(f"The shape of attention out {Attention_rms.shape}")
print(f"The shape of attention out {Attention_rms[0][:5]}")

The shape of attention out torch.Size([6, 212, 768])
The shape of attention out tensor([[1.0091, 1.0485, 1.0346,  ..., 0.9538, 0.9924, 1.0543],
        [1.0060, 1.0462, 1.0382,  ..., 0.9499, 0.9967, 1.0477],
        [1.0105, 1.0486, 1.0447,  ..., 0.9536, 0.9946, 1.0380],
        [1.0066, 1.0460, 1.0415,  ..., 0.9538, 0.9952, 1.0382],
        [1.0022, 1.0483, 1.0377,  ..., 0.9533, 0.9946, 1.0388]],
       grad_fn=<SliceBackward0>)


## MultiHead Attention Block Done

In [14]:
Vector_embeddings.weight.requires_grad_(False)
r2 = Attention_rms.clone()
from collections import OrderedDict
d_1=Attention_rms.shape[-2]
d_2=Attention_rms.shape[-1]

model = torch.nn.Sequential(
OrderedDict(
        [
            ("Linear1", torch.nn.Linear( d_2,d_2*4)),
            ("relu1", torch.nn.GELU()),
            ("Linear2", torch.nn.Linear(d_2*4,d_2)),

        ]
    )
)
ffn_out = model(Attention_rms)
output = ffn_out + r2
output = torch.nn.functional.rms_norm(output,(768,))



## The FFn Layer Done


In [15]:
layer = []
std = 1.0/(768**0.5)
for i in range(5):
    layer.append({
        "w_q" : (torch.rand(768,768,)*std).requires_grad_(True),
        "w_k" : (torch.rand(768,768,)*std).requires_grad_(True),
        "w_v" : (torch.rand(768,768,)*std).requires_grad_(True),
        "ffn" : torch.nn.Sequential(
OrderedDict(
        [
            ("Linear1", torch.nn.Linear( d_2,d_2*4)),
            ("gelu1", torch.nn.GELU()),
            ("Linear2", torch.nn.Linear(d_2*4,d_2))
        ]
    )
)


    })


for i in range(5):
    r1 = output.clone()

    Q = output @ layer[i]["w_q"]
    K = output @ layer[i]["w_k"]
    V = output @ layer[i]["w_v"]



    Q = Q.reshape(output.shape[0],output.shape[1],num_head,dim).transpose(-2,1)
    K = K.reshape(output.shape[0],output.shape[1],num_head,dim).transpose(-2,1)
    V = V.reshape(output.shape[0],output.shape[1],num_head,dim).transpose(-2,1)
    scores = (Q @ K.transpose(-2,-1)) / np.sqrt(dim)
    mask = torch.triu(torch.ones(scores.shape[-1],scores.shape[-1]), diagonal=1).bool()
    scores = scores.masked_fill(mask, float("-inf"))
    Attention_score = torch.nn.functional.softmax( scores,dim=-1)@V
    Attn_out = Attention_score.permute(0, 2, 1, 3).contiguous().reshape(b, t, n*h)
    Attn_out += r1
    Attn_rms = torch.nn.functional.rms_norm(Attn_out,(768,))
    r2 = Attn_rms.clone()


    ffn_out = layer[i]["ffn"](Attn_rms)

    ffn_out = ffn_out + r2

    ffn_out = torch.nn.functional.rms_norm(ffn_out,(768,))
    output = ffn_out





all_params = []

for i in layer:
    all_params += [i["w_k"],i["w_v"],i["w_q"]]
    all_params += list(i["ffn"].parameters())

all_params += [w_k,w_q,w_v]




In [16]:
output.shape

torch.Size([6, 212, 768])

In [17]:

vocab = 50257

output_projection = torch.nn.Linear(768,vocab)

logits = output_projection(output).transpose(-1,-2).to(device)

all_params += list(output_projection.parameters())

ids = tokens["input_ids"].to(device)
target = torch.cat([ids[:,1:],ids[:,-1:]],dim=1)

optimizer = torch.optim.Adam(all_params,lr=1e-2)
loss = torch.nn.functional.cross_entropy(logits,target)




In [18]:
logits.shape,target.shape

(torch.Size([6, 50257, 212]), torch.Size([6, 212]))

In [19]:
losses = []

for j in range(50):
    embeddings = Vector_embeddings(tokens["input_ids"])
    pe = positional_encoding(embeddings.shape[1], embeddings.shape[2])
    input_embeddings = embeddings + pe.repeat(embeddings.shape[0], 1, 1)
    output = input_embeddings

    residual_1 = input_embeddings.clone()
    Q = input_embeddings @ w_q
    K = input_embeddings @ w_k
    V = input_embeddings @ w_v
    Q = Q.reshape(output.shape[0], output.shape[1], num_head, dim).transpose(-2, 1)
    K = K.reshape(output.shape[0], output.shape[1], num_head, dim).transpose(-2, 1)
    V = V.reshape(output.shape[0], output.shape[1], num_head, dim).transpose(-2, 1)
    scores = (Q @ K.transpose(-2, -1)) / np.sqrt(dim)
    mask = torch.triu(torch.ones(scores.shape[-1], scores.shape[-1]), diagonal=1).bool()
    scores = scores.masked_fill(mask, float("-inf"))

    Attention_score = torch.nn.functional.softmax(scores, dim=-1) @ V
    b, n, t, h = Attention_score.shape
    Attention_score = Attention_score.permute(0, 2, 1, 3).contiguous().reshape(b, t, n * h)  # ✅ fix
    output = torch.nn.functional.rms_norm(Attention_score + residual_1, (768,))

    for i in range(5):
        r1 = output.clone()
        Q = output @ layer[i]["w_q"]
        K = output @ layer[i]["w_k"]
        V = output @ layer[i]["w_v"]
        Q = Q.reshape(output.shape[0], output.shape[1], num_head, dim).transpose(-2, 1)
        K = K.reshape(output.shape[0], output.shape[1], num_head, dim).transpose(-2, 1)
        V = V.reshape(output.shape[0], output.shape[1], num_head, dim).transpose(-2, 1)
        scores = (Q @ K.transpose(-2, -1)) / np.sqrt(dim)
        mask = torch.triu(torch.ones(scores.shape[-1], scores.shape[-1]), diagonal=1).bool()
        scores = scores.masked_fill(mask, float("-inf"))

        Attention_score = torch.nn.functional.softmax(scores, dim=-1) @ V
        b, n, t, h = Attention_score.shape
        Attention_score = Attention_score.permute(0, 2, 1, 3).contiguous().reshape(b, t, n * h)  # ✅ fix

        Attn_out = Attention_score + r1
        Attn_rms = torch.nn.functional.rms_norm(Attn_out, (768,))
        r2 = Attn_rms.clone()
        ffn_out = layer[i]["ffn"](Attn_rms)
        ffn_out = torch.nn.functional.rms_norm(ffn_out + r2, (768,))
        output = ffn_out

    logits = output_projection(output).to(device)
    B, T, Voc = logits.shape
    loss = torch.nn.functional.cross_entropy(
        logits.view(B * T, Voc),
        target.view(B * T)
    )

    optimizer.zero_grad()
    loss.backward()          # ✅ fix: removed retain_graph=True (memory leak)
    optimizer.step()
    losses.append(loss.item())

    if j % 10 == 0:
        print(f"Loss at iteration {j}: {loss.item():.4f}")


Loss at iteration 0: 11.1338
Loss at iteration 10: 5.7493
Loss at iteration 20: 5.2409
Loss at iteration 30: 4.8581
Loss at iteration 40: 4.6499


In [20]:
output.shape

torch.Size([6, 212, 768])

In [21]:
def generate(prompt,max_tokens = 20,temp = 1.0):
    tokens = tokenizer(prompt,return_tensors="pt",padding=True,truncation=True)
    ids = tokens["input_ids"]

    for _ in range(max_tokens):
        embeddings = Vector_embeddings(ids)
        pe = positional_encoding(embeddings.shape[1],embeddings.shape[2])
        input_embeddings = embeddings + pe.repeat(embeddings.shape[0],1,1)
        input_embeddings = input_embeddings
        output = input_embeddings

        r1 = input_embeddings.clone()
        Q = input_embeddings @ w_q
        K = input_embeddings @ w_k
        V = input_embeddings @ w_v
        Q = Q.reshape(output.shape[0],output.shape[1], num_head, dim).transpose(-2, 1)
        K = K.reshape(output.shape[0],output.shape[1], num_head, dim).transpose(-2, 1)
        V = V.reshape(output.shape[0],output.shape[1], num_head, dim).transpose(-2, 1)
        scores = (Q @ K.transpose(-2,-1)) / np.sqrt(dim)
        mask = torch.triu(torch.ones(scores.shape[-1],scores.shape[-1]), diagonal=1).bool()
        scores = scores.masked_fill(mask, float("-inf"))

        Attention_score = torch.nn.functional.softmax( scores,dim=-1)@V
        b,n,t,h = Attention_score.shape
        Attn_out = Attention_score.permute(0, 2, 1, 3).contiguous().reshape(b, t, n*h)  + r1

        output = torch.nn.functional.rms_norm(Attn_out, (768,))


        # train
        for i in range(5):

            r1 = output.clone()

            Q = output @ layer[i]["w_q"]
            K = output @ layer[i]["w_k"]
            V = output @ layer[i]["w_v"]

            Q = Q.reshape(output.shape[0],output.shape[1],num_head,dim).transpose(-2,1)
            K = K.reshape(output.shape[0],output.shape[1],num_head,dim).transpose(-2,1)
            V = V.reshape(output.shape[0],output.shape[1],num_head,dim).transpose(-2,1)


            scores = (Q @ K.transpose(-2,-1)) / np.sqrt(dim)
            mask = torch.triu(torch.ones(scores.shape[-1],scores.shape[-1]), diagonal=1).bool()
            scores = scores.masked_fill(mask, float("-inf"))
            Attention_score = torch.nn.functional.softmax( scores,dim=-1)@V
            b,n,t,h = Attention_score.shape
            Attn_out = Attention_score.permute(0, 2, 1, 3).contiguous().reshape(b, t, n*h)  + r1

            output = torch.nn.functional.rms_norm(Attn_out, (768,))

            r2 = Attn_rms.clone()

            ffn_out = layer[i]["ffn"](Attn_rms)

            ffn_out = ffn_out + r2

            ffn_out = torch.nn.functional.rms_norm(ffn_out,(768,))
            output = ffn_out



        logits = output_projection(output[0,-1,:])

        logits = logits/temp

        probs = torch.nn.functional.softmax(logits,dim=-1)

        next_token = torch.multinomial(probs,num_samples=1)

        ids = torch.cat([ids,next_token.unsqueeze(0)],dim=1)

        if next_token == tokenizer.eos_token_id:
            break


    return tokenizer.decode(ids,skip_special_tokens=True)



In [22]:
prompt = "One day,"

ans = generate(prompt,max_tokens=20,temp=0.5)
ans1 = generate(prompt,max_tokens=20,temp=0.8)
ans2 = generate(prompt,max_tokens=20,temp=0.1)
ans3 = generate(prompt,max_tokens=20,temp=1.2)

In [23]:
ans

['One day, to']

In [24]:
ans1

['One day, her']

In [25]:
ans2

['One day,']

In [26]:
ans3

['One day, hug So this The It He the kay and could driving fish littleep day\n its and her and']